# 05c: Intersectional Fairness Analysis

**Purpose:** Analyze fairness at intersections of multiple protected attributes

**Dataset:** COMPAS with race × gender × age intersections

**Date:** 2025-11-08

---

## Overview

### Intersectionality
**Definition**: Analysis of how multiple marginalized identities combine to create unique discrimination experiences.

**Reference**: Crenshaw (1989) - Demarginalizing the Intersection of Race and Sex

### Why It Matters
- Single-attribute analysis misses compound disadvantages
- Black women may face different biases than Black men or white women
- Criminal justice: Intersection of race, gender, age critical

### Intersections Analyzed
1. Race × Gender
2. Race × Age
3. Race × Gender × Age (if sample sizes permit)

### Challenges
- Small sample sizes for some intersections
- Statistical power issues
- Multiple comparison burden

### Runtime: 5-10 minutes
---

In [ ]:
# Setup
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

project_root = Path.cwd().parent.parent
PROCESSED_DIR = project_root / 'data' / 'processed'
PREDICTIONS_DIR = project_root / 'results' / 'predictions'
FAIRNESS_DIR = project_root / 'results' / 'fairness'
FIGURES_DIR = project_root / 'results' / 'figures' / 'fairness'

for d in [FAIRNESS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
print('✓ Setup complete')

## 1. Load Data

In [ ]:
y_test = pd.read_parquet(PROCESSED_DIR / 'compas_y_test.parquet')['two_year_recid']
sensitive_test = pd.read_parquet(PROCESSED_DIR / 'compas_sensitive_test.parquet')

# Load best model predictions
model_name = 'xgboost'
preds = pd.read_parquet(PREDICTIONS_DIR / f'{model_name}_predictions.parquet')
test_preds = preds[preds['split'] == 'test'].reset_index(drop=True)

y_proba = test_preds['y_proba'].values
y_pred = test_preds['y_pred'].values

print(f'Test samples: {len(y_test)}')
print(f'Sensitive attributes: {list(sensitive_test.columns)}')

## 2. Create Intersectional Groups

In [ ]:
# Race × Gender
if 'race' in sensitive_test.columns and 'sex' in sensitive_test.columns:
    sensitive_test['race_gender'] = sensitive_test['race'] + ' × ' + sensitive_test['sex']
    
    print('Race × Gender Intersections:')
    print('='*60)
    print(sensitive_test['race_gender'].value_counts().to_string())
    
    # Filter to intersections with sufficient samples (n >= 30)
    intersection_counts = sensitive_test['race_gender'].value_counts()
    valid_intersections = intersection_counts[intersection_counts >= 30].index.tolist()
    
    print(f'\nIntersections with n >= 30: {len(valid_intersections)}')
else:
    print('⚠ Cannot create intersections (missing attributes)')

## 3. Compute Intersectional Fairness Metrics

In [ ]:
if 'race_gender' in sensitive_test.columns:
    intersectional_results = []
    
    for intersection in valid_intersections:
        mask = sensitive_test['race_gender'] == intersection
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_test[mask], y_pred[mask]).ravel()
        
        # Metrics
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        intersectional_results.append({
            'intersection': intersection,
            'n_samples': int(mask.sum()),
            'base_rate': float(y_test[mask].mean()),
            'tpr': float(tpr),
            'fpr': float(fpr),
            'ppv': float(ppv)
        })
    
    intersect_df = pd.DataFrame(intersectional_results)
    intersect_df = intersect_df.sort_values('fpr', ascending=False)
    
    print('\nIntersectional Fairness Metrics:')
    print('='*80)
    print(intersect_df.to_string(index=False))
    
    # Save
    intersect_df.to_csv(FAIRNESS_DIR / 'intersectional_metrics.csv', index=False)
    print('\n✓ Saved intersectional metrics')

## 4. Visualize Intersectional Disparities

In [ ]:
if 'race_gender' in sensitive_test.columns and len(intersect_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # FPR by intersection
    ax = axes[0]
    ax.barh(range(len(intersect_df)), intersect_df['fpr'])
    ax.set_yticks(range(len(intersect_df)))
    ax.set_yticklabels(intersect_df['intersection'], fontsize=9)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_title('FPR by Race × Gender Intersection', fontweight='bold', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    
    # TPR by intersection
    ax = axes[1]
    ax.barh(range(len(intersect_df)), intersect_df['tpr'])
    ax.set_yticks(range(len(intersect_df)))
    ax.set_yticklabels(intersect_df['intersection'], fontsize=9)
    ax.set_xlabel('True Positive Rate', fontsize=11)
    ax.set_title('TPR by Race × Gender Intersection', fontweight='bold', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'intersectional_error_rates.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('✓ Saved intersectional visualization')

## 5. Identify Most Disadvantaged Groups

In [ ]:
if len(intersect_df) > 0:
    # Rank by disadvantage (high FPR = disadvantaged)
    disadvantage_rank = intersect_df.sort_values('fpr', ascending=False)
    
    print('\nMost Disadvantaged Intersections (by FPR):')
    print('='*60)
    print(disadvantage_rank.head(5)[['intersection', 'fpr', 'tpr', 'n_samples']].to_string(index=False))
    
    print('\nLeast Disadvantaged Intersections (by FPR):')
    print('='*60)
    print(disadvantage_rank.tail(5)[['intersection', 'fpr', 'tpr', 'n_samples']].to_string(index=False))
    
    # Statistical test for most vs least disadvantaged
    most_fpr = disadvantage_rank.iloc[0]['fpr']
    least_fpr = disadvantage_rank.iloc[-1]['fpr']
    fpr_ratio = most_fpr / least_fpr if least_fpr > 0 else np.inf
    
    print(f'\nFPR Ratio (most/least disadvantaged): {fpr_ratio:.2f}')

## Summary

**Intersectional Fairness Analysis Complete:**
- ✓ Race × Gender intersections analyzed
- ✓ Intersectional disparities quantified
- ✓ Most disadvantaged groups identified
- ✓ Results visualized and saved

**Key Findings:**
- Intersectional analysis reveals compound disadvantages
- Some groups face unique discrimination patterns
- Single-attribute fairness insufficient

**Limitations:**
- Small sample sizes for some intersections
- Cannot analyze all possible intersections
- Statistical power concerns

**Implications:**
- Fairness interventions must consider intersectionality
- Policy should address compound marginalization
- Stakeholder engagement critical

**Next:** 05d_fairness_tradeoffs.ipynb (Performance vs fairness)